# 00. Setup Inicial: Unity Catalog, Schemas, Volume y Seguridad
**Proyecto Integrador: ElectroCasa**

Este notebook inicializa la infraestructura base en Unity Catalog para el proyecto:
1. Creación del catálogo del proyecto (`electrocasa_dev` o `electrocasa_prod`).
2. Creación de los schemas por capa Medallion (`bronze`, `silver`, `gold`).
3. Creación del Volume de landing (`default.vol_landing`) y sus subcarpetas por fuente.
4. Configuración de gobernanza: Permisos `GRANT` para los 3 grupos de trabajo.
5. Demostración de `REVOKE` con evidencia de `SHOW GRANTS`.
6. Creación de funciones de enmascaramiento dinámico (Data Masking) para `dni` y `salario`.
7. Prueba de permisos sobre el catálogo `system` para auditoría.

In [2]:
# Databricks runtime
from databricks.sdk.runtime import spark, dbutils

In [ ]:
# Widget interactivo para parametrizar el nombre del catálogo según el target (dev o prod)
dbutils.widgets.text("catalogo", "electrocasa_dev", "Nombre del Catálogo")
catalogo = dbutils.widgets.get("catalogo")
print(f"Configurando entorno para el catálogo: {catalogo}")

In [ ]:
# 1. Creación del Catálogo y Schemas por Capa Medallion
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print(f"Catálogo {catalogo} y esquemas bronze, silver y gold creados/verificados correctamente.")

In [ ]:
# 2. Creación del Volume de Landing en el esquema default
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.default.vol_landing")

# Subcarpetas obligatorias para cada fuente de datos
fuentes = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]
for f in fuentes:
    path = f"/Volumes/{catalogo}/default/vol_landing/{f}"
    dbutils.fs.mkdirs(path)
    print(f"Directorio verificado: {path}")

### 3. Gobierno y Seguridad: Asignación de Roles y Permisos (GRANT)
Se configuran los 3 grupos requeridos:
- `electrocasa_ingenieria`: Control total sobre bronze, silver y gold.
- `electrocasa_analistas`: Solo lectura sobre gold.
- `electrocasa_auditoria`: Solo lectura sobre gold y linaje.

In [ ]:
# Asignación de permisos a grupos
# Nota: Los grupos deben haber sido creados en Settings > Identity and access > Groups
try:
    # Permiso de uso del catálogo a los 3 grupos
    spark.sql(f"GRANT USE CATALOG ON CATALOG {catalogo} TO `electrocasa_ingenieria`")
    spark.sql(f"GRANT USE CATALOG ON CATALOG {catalogo} TO `electrocasa_analistas`")
    spark.sql(f"GRANT USE CATALOG ON CATALOG {catalogo} TO `electrocasa_auditoria`")

    # electrocasa_ingenieria: Uso y modificación sobre bronze, silver y gold
    for sch in ["bronze", "silver", "gold", "default"]:
        spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA {catalogo}.{sch} TO `electrocasa_ingenieria`")

    # electrocasa_analistas y electrocasa_auditoria: Solo lectura sobre gold
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalogo}.gold TO `electrocasa_analistas`")
    spark.sql(f"GRANT SELECT ON SCHEMA {catalogo}.gold TO `electrocasa_analistas`")

    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalogo}.gold TO `electrocasa_auditoria`")
    spark.sql(f"GRANT SELECT ON SCHEMA {catalogo}.gold TO `electrocasa_auditoria`")
    print("Permisos GRANT aplicados con éxito a los grupos.")
except Exception as e:
    print("Aviso al asignar permisos a grupos (probable Free Edition sin grupos creados):")
    print(e)
    print("Fallback: Asegúrate de crear los grupos en Settings > Identity and access > Groups")

### 4. Demostración de REVOKE
Asignamos temporalmente a `electrocasa_analistas` un permiso indebido sobre `silver`, verificamos con `SHOW GRANTS`, luego lo revocamos y comprobamos nuevamente.

In [ ]:
try:
    print("--- 1. Otorgando permiso indebido a analistas sobre silver ---")
    spark.sql(f"GRANT SELECT ON SCHEMA {catalogo}.silver TO `electrocasa_analistas`")
    display(spark.sql(f"SHOW GRANTS ON SCHEMA {catalogo}.silver"))

    print("--- 2. Revocando el permiso indebido (REVOKE) ---")
    spark.sql(f"REVOKE SELECT ON SCHEMA {catalogo}.silver FROM `electrocasa_analistas`")
    display(spark.sql(f"SHOW GRANTS ON SCHEMA {catalogo}.silver"))
    print("REVOKE completado y verificado exitosamente.")
except Exception as e:
    print("Error en demostración de REVOKE:", e)

### 5. Funciones de Enmascaramiento Dinámico (Data Masking)
Permite que solo los miembros de `electrocasa_ingenieria` vean el `dni` y `salario` reales en la capa Silver/Gold, enmascarando los valores para analistas y auditoría.

In [ ]:
spark.sql(f"USE CATALOG {catalogo}")
spark.sql("USE SCHEMA silver")

spark.sql("""
CREATE OR REPLACE FUNCTION mask_dni(val STRING)
RETURNS STRING
RETURN CASE 
    WHEN is_account_group_member('electrocasa_ingenieria') THEN val
    ELSE concat('***-', right(val, 2))
END
""")

spark.sql("""
CREATE OR REPLACE FUNCTION mask_salario(val DOUBLE)
RETURNS DOUBLE
RETURN CASE 
    WHEN is_account_group_member('electrocasa_ingenieria') THEN val
    ELSE -1.0
END
""")

print("Funciones mask_dni y mask_salario creadas exitosamente en silver.")

### 6. Acceso de Auditoría al Catálogo System (Evidencia Free Edition)
Intentamos otorgar acceso a las tablas de auditoría de `system`. En Free Edition este comando fallará por falta de privilegios `MANAGE` sobre `system`, lo cual documentamos en el README.

In [ ]:
try:
    spark.sql("GRANT USE CATALOG ON CATALOG system TO `electrocasa_auditoria`")
    spark.sql("GRANT USE SCHEMA ON SCHEMA system.access TO `electrocasa_auditoria`")
    spark.sql("GRANT SELECT ON TABLE system.access.audit TO `electrocasa_auditoria`")
    print("Permisos sobre system otorgados exitosamente.")
except Exception as e:
    print("Comportamiento esperado en Databricks Free Edition:")
    print("No se tienen permisos de administración sobre el catálogo system.")
    print("Error:", e)
    print("Alternativa oficial: Auditoría visual vía Catalog Explorer > Lineage / Event Log.")